In [1]:
import torch
import tiktoken

In [3]:
EMB_SIZE = 6
tokenizer_gpt2 = tiktoken.get_encoding("gpt2")

In [6]:
class TestModel(torch.nn.Module):
    def __init__(self):
        super().__init__()
        self.layer = torch.nn.Sequential(
            torch.nn.Embedding(tokenizer_gpt2.n_vocab, EMB_SIZE),
            torch.nn.Linear(EMB_SIZE, tokenizer_gpt2.n_vocab),
        )

    def forward(self, x):
        return self.layer(x)

In [11]:
test_text = "Suno Dost, Shah Nawaz "
temperature = 5
test_text_encoded = tokenizer_gpt2.encode(test_text)
t_model = TestModel()
t_model.eval()
with torch.no_grad():
    for _ in range(10):
        out_logits = t_model(torch.tensor([test_text_encoded]))
        # last token as next word
        out_logits = out_logits.squeeze()[-1]

        # 1. Top K
        top_k_logits, top_k_pos = torch.topk(out_logits, k=5)
        new_logits = torch.where(
            condition=out_logits < top_k_logits[-1],
            input=torch.tensor(float("-inf")),
            other=out_logits,
        )

        # 2. Temperature Scaling
        out_logits = new_logits / temperature
        out_logits = torch.softmax(out_logits, dim=-1)
        next_token = torch.multinomial(out_logits, num_samples=1).item()

        # 3. Greedy Decoding
        # next_token = torch.argmax(out_logits)

        next_token = [next_token]
        test_text_encoded += next_token
        next_token_decoded = tokenizer_gpt2.decode(next_token)
        test_text += next_token_decoded

print(test_text)

Suno Dost, Shah Nawaz ais sailor)); fuRich winds Tier Levine Dress interacted
